# Praktikum Sub-CPMK-5 &mdash; Restorasi Citra dan Operasi Dasar Geometrik

**Pengolahan Citra Digital (IF2514505)** &nbsp;|&nbsp; Semester 5 &nbsp;|&nbsp; 1 SKS Praktikum &nbsp;|&nbsp; Durasi sasaran 135 menit (rentang 120&ndash;150)

**Program Studi Informatika &mdash; Institut Teknologi Kalimantan**

---

| | |
|---|---|
| **Nama** | ................................................ |
| **NIM** | ................................................ |
| **Kelas / Kelompok** | ................................................ |
| **Tanggal praktikum** | ................................................ |

---

### Capaian yang diukur
**Sub-CPMK-5**: Mahasiswa mampu menerapkan teknik restorasi citra dan operasi dasar pengolahan citra digital.
Indikator: (1) menjelaskan model degradasi dan model derau serta menerapkan penapis restorasi (inverse, Wiener); (2) menerapkan operasi dasar (warping, cropping, zooming) beserta metode interpolasi yang sesuai.

### Rujukan
Gonzalez, R.C. dan Woods, R.E. (2018) *Digital Image Processing*, edisi ke-4: Bab 5 (§5.1&ndash;5.9) untuk restorasi, §2.4 dan §2.6 untuk interpolasi dan transformasi geometrik. Nomor persamaan mengikuti buku tersebut.

### Petunjuk pengerjaan
1. Jalankan sel berurutan. Sel bertanda `# TODO` wajib Anda lengkapi sendiri.
2. Setiap blok **Analisis** diisi kalimat Anda sendiri dan **wajib memuat angka** dari keluaran Anda.
3. Citra uji dibangkitkan di notebook (benih tetap; huruf memakai fon bitmap 5&times;7 yang didefinisikan di notebook). Foto koin diambil dari `skimage.data`. **Tidak perlu mengunggah berkas.**
4. Koordinat: pada kode ini **x = kolom, y = baris** (konvensi OpenCV). Gonzalez memakai x = baris; matriksnya setara setelah sumbu ditukar.
5. Fungsi `baca(citra, kotak)` adalah pembaca huruf sederhana berbasis templat pada **posisi kotak yang sudah diketahui**. Ia mengukur apakah huruf dapat dikenali di tempat yang seharusnya, bukan OCR umum.
6. Bagian **[Pengayaan]** boleh dilewati; jalur inti selesai dalam &plusmn;120 menit.
7. Kumpulkan dalam keadaan **sudah dijalankan penuh** dengan nama `PraktikumPCD_SubCPMK5_NIM_Nama.ipynb`.

| Bagian | Isi | Menit |
|---|---|---|
| 0 | Persiapan dan citra uji | 10 |
| 1 | Model derau dan menaksir parameternya (§5.2) | 15 |
| 2 | Restorasi bila hanya ada derau: penapis rerata, statistik urutan, adaptif (§5.3) | 25 |
| 3 | Fungsi degradasi, penapis invers, Wiener (§5.5&ndash;5.8) | 30 |
| 4 | Interpolasi, pemetaan mundur, perbesaran, penyusutan, rotasi (§2.4, §2.6) | 25 |
| 5 | Studi kasus: meluruskan foto manifes yang miring (§2.6, registrasi) | 20 |
| 6 | Pemeriksaan akhir dan pengumpulan | 10 |


## Bagian 0 &mdash; Persiapan (10 menit)
Jalankan sel berikut **tanpa diubah**.

In [ ]:
# Sel persiapan — jalankan tanpa diubah
import numpy as np
import cv2
import matplotlib.pyplot as plt
from numpy.lib.stride_tricks import sliding_window_view
from scipy import ndimage
from skimage import data as skdata

plt.rcParams['figure.dpi'] = 110
print('NumPy', np.__version__, '| OpenCV', cv2.__version__)

"""Pembangkit citra uji sintetis Sub-CPMK-5 (benih acak tetap, hanya NumPy).

Huruf memakai fon bitmap 5x7 yang didefinisikan di sini, sehingga citra identik di semua versi pustaka.
"""
import numpy as np

BENIH = 2027

_FON = {
    "A": ".###.|#...#|#...#|#####|#...#|#...#|#...#", "B": "####.|#...#|#...#|####.|#...#|#...#|####.",
    "C": ".###.|#...#|#....|#....|#....|#...#|.###.", "D": "####.|#...#|#...#|#...#|#...#|#...#|####.",
    "E": "#####|#....|#....|####.|#....|#....|#####", "F": "#####|#....|#....|####.|#....|#....|#....",
    "G": ".###.|#...#|#....|#.###|#...#|#...#|.###.", "H": "#...#|#...#|#...#|#####|#...#|#...#|#...#",
    "I": ".###.|..#..|..#..|..#..|..#..|..#..|.###.", "J": "..###|...#.|...#.|...#.|...#.|#..#.|.##..",
    "K": "#...#|#..#.|#.#..|##...|#.#..|#..#.|#...#", "L": "#....|#....|#....|#....|#....|#....|#####",
    "M": "#...#|##.##|#.#.#|#.#.#|#...#|#...#|#...#", "N": "#...#|#...#|##..#|#.#.#|#..##|#...#|#...#",
    "O": ".###.|#...#|#...#|#...#|#...#|#...#|.###.", "P": "####.|#...#|#...#|####.|#....|#....|#....",
    "Q": ".###.|#...#|#...#|#...#|#.#.#|#..#.|.##.#", "R": "####.|#...#|#...#|####.|#.#..|#..#.|#...#",
    "S": ".####|#....|#....|.###.|....#|....#|####.", "T": "#####|..#..|..#..|..#..|..#..|..#..|..#..",
    "U": "#...#|#...#|#...#|#...#|#...#|#...#|.###.", "V": "#...#|#...#|#...#|#...#|#...#|.#.#.|..#..",
    "W": "#...#|#...#|#...#|#.#.#|#.#.#|#.#.#|.#.#.", "X": "#...#|#...#|.#.#.|..#..|.#.#.|#...#|#...#",
    "Y": "#...#|#...#|.#.#.|..#..|..#..|..#..|..#..", "Z": "#####|....#|...#.|..#..|.#...|#....|#####",
    "0": ".###.|#...#|#..##|#.#.#|##..#|#...#|.###.", "1": "..#..|.##..|..#..|..#..|..#..|..#..|.###.",
    "2": ".###.|#...#|....#|...#.|..#..|.#...|#####", "3": "#####|...#.|..#..|...#.|....#|#...#|.###.",
    "4": "...#.|..##.|.#.#.|#..#.|#####|...#.|...#.", "5": "#####|#....|####.|....#|....#|#...#|.###.",
    "6": "..##.|.#...|#....|####.|#...#|#...#|.###.", "7": "#####|....#|...#.|..#..|.#...|.#...|.#...",
    "8": ".###.|#...#|#...#|.###.|#...#|#...#|.###.", "9": ".###.|#...#|#...#|.####|....#|...#.|.##..",
    " ": ".....|.....|.....|.....|.....|.....|.....",
}
GLIF = {k: np.array([[c == "#" for c in r] for r in v.split("|")], float) for k, v in _FON.items()}
SKALA, LANGKAH = 3, 18          # piksel per titik fon, jarak antarhuruf (piksel)


def tulis(img, teks, baris, kolom, nilai, skala=SKALA, langkah=LANGKAH):
    """Menulis teks pada img (float) mulai (baris, kolom); mengembalikan daftar kotak huruf (b0, k0, karakter)."""
    kotak = []
    for i, ch in enumerate(teks):
        g = np.kron(GLIF[ch], np.ones((skala, skala)))
        b0, k0 = baris, kolom + i * langkah
        sub = img[b0:b0 + g.shape[0], k0:k0 + g.shape[1]]
        sub[g > 0] = nilai
        kotak.append((b0, k0, ch))
    return kotak


def baca(img, kotak, skala=SKALA, teks_gelap=False):
    """Pembaca huruf sederhana: korelasi ternormalisasi tiap kotak dengan semua glif. Mengembalikan (teks, akurasi).

    teks_gelap=True untuk huruf gelap di atas latar terang (citra dibalik dulu).
    """
    h, w = 7 * skala, 5 * skala
    cal = {k: np.kron(v, np.ones((skala, skala))) for k, v in GLIF.items() if k != " "}
    cal = {k: (v - v.mean()) / v.std() for k, v in cal.items()}
    hasil, benar, total = [], 0, 0
    for b0, k0, ch in kotak:
        if ch == " ":
            hasil.append(" "); continue
        p = img[b0:b0 + h, k0:k0 + w].astype(float)
        p = -p if teks_gelap else p
        p = (p - p.mean()) / (p.std() + 1e-9)
        tebak = max(cal, key=lambda k: float((p * cal[k]).mean()))
        hasil.append(tebak); total += 1; benar += tebak == ch
    return "".join(hasil), benar / max(total, 1)


def _ke_uint8(x):
    return np.clip(np.floor(x + 0.5), 0, 255).astype(np.uint8)


def pola_uji(h=256, w=256):
    """Empat aras keabuan (40, 100, 160, 220) seperti pola uji Gonzalez Gbr. 5.3, untuk percobaan derau."""
    yy, xx = np.mgrid[0:h, 0:w]
    img = np.full((h, w), 40.0)
    img[24:232, 24:232] = 100
    img[(yy - 128) ** 2 + (xx - 128) ** 2 <= 72 ** 2] = 160
    img[96:160, 96:160] = 220
    return img.astype(np.uint8)


TEKS_LAMBUNG = ["KM NUSA BAHARI 07", "IMO 9183746 GT 512"]


def papan_lambung(h=128, w=352):
    """Tulisan putih pada lambung kapal gelap (float 0..255) beserta kotak huruf untuk pembaca."""
    img = np.full((h, w), 55.0)
    img[:, :] += np.linspace(0, 20, w)[None, :]          # gradasi cat lambung
    img[6:10, :] = 120; img[h - 10:h - 6, :] = 120         # garis lambung
    kotak = tulis(img, TEKS_LAMBUNG[0], 24, 23, 225) + tulis(img, TEKS_LAMBUNG[1], 74, 14, 225)
    return img, kotak


def psf_gerak(h, w, L=15):
    """PSF kabur gerak mendatar sepanjang L piksel, ditempatkan di titik asal (konvolusi sirkular)."""
    p = np.zeros((h, w)); p[0, :L] = 1.0 / L
    return np.roll(p, -(L // 2), axis=1)


def psf_gauss(h, w, s=2.0):
    yy, xx = np.mgrid[0:h, 0:w]
    yy = np.minimum(yy, h - yy); xx = np.minimum(xx, w - xx)
    p = np.exp(-(yy ** 2 + xx ** 2) / (2 * s * s))
    return p / p.sum()


def kaburkan(f, psf):
    """g = h ∗ f dengan konvolusi sirkular (model Eq. 5-1 tanpa derau)."""
    return np.real(np.fft.ifft2(np.fft.fft2(f) * np.fft.fft2(psf)))


TEKS_MANIFES = ["MANIFES KM NUSA 07", "01 ARIF 34", "02 SINTA 27", "03 BUDI 45", "04 RINA 19", "05 DODI 52", "06 MAYA 08"]


def papan_manifes(h=330, w=360):
    """Lembar manifes acuan (tegak lurus kamera): teks hitam di atas kertas putih, bergaris."""
    img = np.full((h, w), 235.0)
    kotak = []
    for i, t in enumerate(TEKS_MANIFES):
        b = 20 + i * 44
        kotak += tulis(img, t, b, 18, 30)
        if i > 0:
            img[b + 28:b + 30, 12:w - 12] = 150
    img[:4, :] = img[-4:, :] = 90; img[:, :4] = img[:, -4:] = 90
    return img, kotak


SUDUT_MIRING = np.array([[70.0, 62.0], [400.0, 18.0], [445.0, 372.0], [40.0, 300.0]])   # (x=kolom, y=baris): kiri-atas, kanan-atas, kanan-bawah, kiri-bawah


def manifes_miring(rng, sigma=3.0, H_out=400, W_out=480):
    """Foto miring lembar manifes: homografi dari acuan ke sudut SUDUT_MIRING, di atas meja gelap, berderau."""
    ref, _ = papan_manifes()
    h, w = ref.shape
    src = np.array([[0, 0], [w - 1, 0], [w - 1, h - 1], [0, h - 1]], float)
    Hm = homografi(src, SUDUT_MIRING)
    Hi = np.linalg.inv(Hm)
    yy, xx = np.mgrid[0:H_out, 0:W_out].astype(float)
    P = Hi @ np.stack([xx.ravel(), yy.ravel(), np.ones(xx.size)])
    u, v = P[0] / P[2], P[1] / P[2]
    out = np.full(H_out * W_out, 60.0)
    ok = (u >= 0) & (u <= w - 1) & (v >= 0) & (v <= h - 1)
    u0, v0 = np.floor(u[ok]).astype(int), np.floor(v[ok]).astype(int)
    u1, v1 = np.minimum(u0 + 1, w - 1), np.minimum(v0 + 1, h - 1)
    a, b = u[ok] - u0, v[ok] - v0
    out[ok] = (1 - a) * (1 - b) * ref[v0, u0] + a * (1 - b) * ref[v0, u1] + (1 - a) * b * ref[v1, u0] + a * b * ref[v1, u1]
    out = out.reshape(H_out, W_out) + rng.normal(0, sigma, (H_out, W_out))
    return _ke_uint8(out)


def homografi(src, dst):
    """Matriks 3x3 yang memetakan 4 titik src (x, y) ke dst (x, y) (sistem linear 8 peubah)."""
    A, bb = [], []
    for (x, y), (u, v) in zip(src, dst):
        A.append([x, y, 1, 0, 0, 0, -u * x, -u * y]); bb.append(u)
        A.append([0, 0, 0, x, y, 1, -v * x, -v * y]); bb.append(v)
    h = np.linalg.solve(np.array(A, float), np.array(bb, float))
    return np.append(h, 1).reshape(3, 3)


def kisi_halus(h=256, w=256, periode=3.0):
    """Kisi kosinus berperiode 3 piksel (dekat batas Nyquist) untuk percobaan aliasing saat menyusutkan."""
    xx = np.mgrid[0:h, 0:w][1]
    return _ke_uint8(128 + 100 * np.cos(2 * np.pi * xx / periode))


def bangkitkan_semua():
    rng = np.random.default_rng(BENIH)
    lambung, kotak_l = papan_lambung()
    manifes, kotak_m = papan_manifes()
    return {"pola": pola_uji(), "lambung": lambung, "kotak_lambung": kotak_l,
            "manifes_acuan": _ke_uint8(manifes), "kotak_manifes": kotak_m,
            "manifes_miring": manifes_miring(rng), "kisi": kisi_halus()}

def tampil(daftar, judul=None, kolom=None, tinggi=3.0):
    n = len(daftar); kolom = kolom or n; baris = int(np.ceil(n / kolom))
    fig, ax = plt.subplots(baris, kolom, figsize=(3.8 * kolom, tinggi * baris)); ax = np.atleast_1d(ax).ravel()
    for a in ax: a.axis('off')
    for a, im, t in zip(ax, daftar, judul or [''] * n):
        a.imshow(np.clip(im, 0, 255), cmap='gray', vmin=0, vmax=255, interpolation='nearest'); a.set_title(t, fontsize=9)
    plt.tight_layout(); plt.show()

def psnr(acuan, uji):
    mse = np.mean((np.asarray(acuan, float) - np.clip(np.asarray(uji, float), 0, 255)) ** 2)
    return float('inf') if mse == 0 else float(10 * np.log10(255 ** 2 / mse))

def ke_uint8(x):
    return np.clip(np.floor(np.asarray(x, float) + 0.5), 0, 255).astype(np.uint8)

def jendela(g, m):
    """Semua tetangga m x m setiap piksel (tepi dicerminkan): larik (baris, kolom, m*m)."""
    p = np.pad(np.asarray(g, float), m // 2, mode='reflect')
    return sliding_window_view(p, (m, m)).reshape(g.shape[0], g.shape[1], m * m)

C = bangkitkan_semua()
pola, lambung, kotak_lambung = C['pola'], C['lambung'], C['kotak_lambung']
acuan, kotak_manifes, miring, kisi = C['manifes_acuan'], C['kotak_manifes'], C['manifes_miring'], C['kisi']
koin = skdata.coins()[:300, :384]
print('baca(lambung):', baca(lambung, kotak_lambung))
tampil([pola, lambung, acuan, miring, kisi[:64, :64], koin], ['pola', 'lambung', 'manifes acuan', 'manifes miring', 'kisi (potongan)', 'koin'], kolom=3)

## Bagian 1 &mdash; Model derau dan menaksir parameternya (15 menit)

Model degradasi Gonzalez: $g(x,y) = (h \star f)(x,y) + \eta(x,y)$ (Eq. 5-1), atau $G = HF + N$ (Eq. 5-2). Bagian 1&ndash;2 hanya memuat derau ($h$ = impuls).
Fungsi `derau` di bawah membangkitkan enam PDF Gonzalez (Eq. 5-3 s.d. 5-16).

In [ ]:
# Diberikan — jalankan
def derau(img, jenis, rng, **p):
    f = np.asarray(img, float)
    if jenis == 'gauss':          g = f + rng.normal(p.get('mean', 0), p['sigma'], f.shape)
    elif jenis == 'rayleigh':     g = f + p.get('a', 0) + rng.rayleigh(np.sqrt(p['b'] / 2), f.shape)
    elif jenis == 'erlang':       g = f + rng.gamma(p['b'], 1 / p['a'], f.shape)
    elif jenis == 'eksponensial': g = f + rng.exponential(1 / p['a'], f.shape)
    elif jenis == 'seragam':      g = f + rng.uniform(p['a'], p['b'], f.shape)
    elif jenis == 'garam_merica':
        g = f.copy(); r = rng.random(f.shape)
        g[r < p.get('pp', 0)] = 0
        g[(r >= p.get('pp', 0)) & (r < p.get('pp', 0) + p.get('ps', 0))] = 255
    return ke_uint8(g)

MODEL = {'gauss': dict(sigma=15.0), 'rayleigh': dict(b=1048.5), 'erlang': dict(a=0.0943, b=2),
         'eksponensial': dict(a=1 / 15), 'seragam': dict(a=-26.0, b=26.0), 'garam_merica': dict(ps=0.05, pp=0.05)}


In [ ]:
# TODO 1.1 — Bangkitkan keenam derau pada `pola` dengan rng = np.random.default_rng(11), berurutan seperti MODEL.
#            Pada tambalan datar pola[30:90, 30:60] (aras asli 100), hitung rerata dan simpangan baku (Eq. 5-19, 5-20),
#            lalu bandingkan dengan teori: Rayleigh rerata a + sqrt(πb/4), simp sqrt(b(4−π)/4); Erlang b/a dan sqrt(b)/a;
#            eksponensial 1/a dan 1/a; seragam (a+b)/2 dan (b−a)/sqrt(12). Tampilkan citra dan histogram tambalannya.
rng = np.random.default_rng(11)
denoised1 = {}
for nama, prm in MODEL.items():
    denoised1[nama] = derau(pola, nama, rng, **prm)

patch_asli = pola[30:90, 30:60].astype(float)
print(f"{'derau':14s} {'rerata':>9s} {'std':>9s} | {'teori tambahan':>11s} {'teori std':>9s}")
for nama, prm in MODEL.items():
    g = denoised1[nama]
    pp = g[30:90, 30:60]
    m = float(pp.mean()); s = float(pp.std())
    if nama == 'gauss':
        mt, st = prm.get('mean', 0.0), prm['sigma']
    elif nama == 'rayleigh':
        b = prm['b']
        mt = float(np.sqrt(np.pi * b / 4.0))
        st = float(np.sqrt(b * (4.0 - np.pi) / 4.0))
    elif nama == 'erlang':
        mt = prm['b'] / prm['a']
        st = float(np.sqrt(prm['b']) / prm['a'])
    elif nama == 'eksponensial':
        mt = 1.0 / prm['a']
        st = mt
    elif nama == 'seragam':
        mt = (prm['a'] + prm['b']) / 2.0
        st = (prm['b'] - prm['a']) / np.sqrt(12.0)
    else:
        mt, st = float('nan'), float('nan')
    print(f"{nama:14s} {m:9.2f} {s:9.2f} | {mt:11.2f} {st:9.2f}")

tampil([pola] + [denoised1[k] for k in MODEL],
       ['asli'] + list(MODEL.keys()), kolom=3)
for nama in MODEL:
    pp = denoised1[nama][30:90, 30:60]
    plt.figure(figsize=(5, 2.5))
    plt.hist(pp.ravel(), bins=50, range=(0, 255), color='steelblue', edgecolor='black')
    plt.title(f'Histogram tambalan — {nama} (mean={pp.mean():.1f}, std={pp.std():.1f})')
    plt.xlabel('Nilai piksel'); plt.ylabel('Jumlah'); plt.tight_layout(); plt.show()


**Analisis 1.** (a) Model derau mana yang hampir tidak dapat dibedakan secara visual tetapi jelas berbeda pada histogram tambalan? (b) Mengapa rerata tambalan untuk Rayleigh, Erlang, dan eksponensial bergeser dari 100, dan apa akibatnya bagi penapis yang menganggap derau berrerata nol? (c) Mengapa taksiran Rayleigh dan Erlang sedikit menyimpang dari teori? (Petunjuk: lihat fraksi piksel 255.)

*Jawaban:* (a) Derau seragam hampir terlihat sama seperti citra asli, tetapi histogram tambalannya menunjukkan sebaran datar dari -26 hingga +26 di sekitar 100, berbeda dengan distribusi berbentuk lonceng dari Gaussian. (b) Rayleigh, Erlang, dan eksponensial adalah derau positif: selalu menambahkan nilai ke piksel asli, sehingga rerata tambalan bergeser ke atas dari 100 (mis. eksponensial menambahkan rata-rata 15, jadi rerata circa 115). Penapis yang mengasumsikan derau rata nol (seperti rata-rata aritmetik) akan menghasilkan citra yang terlalu terang karena tidak mengakibatkan kompensasi bagi geseran ini. (c) Taksiran sedikit menyimpang karena piksel yang jadi 0 atau 255 (batasan uint8) tidak lagi mengikuti distribusi teoritis murni; pemotongan di batas 0 dan 255 mengubah rerata dan simpangan baku yang diamati.


## Bagian 2 &mdash; Restorasi bila hanya ada derau (25 menit)

Gonzalez §5.3: rerata aritmetik (Eq. 5-23), geometrik (5-24), harmonik (5-25), kontraharmonik orde $Q$ (5-26):
$$\hat f = \frac{\sum g^{Q+1}}{\sum g^{Q}},$$
median (5-27), maks, min, titik tengah (5-28 s.d. 5-30), dan rerata terpangkas alfa (5-31): buang $d/2$ nilai terendah dan $d/2$ tertinggi, ratakan sisanya.
Semua penapis memakai `jendela(g, m)`. Tambahkan bilangan kecil (1e-6) sebelum log atau pangkat negatif.

In [ ]:
# Diberikan — derau untuk seluruh Bagian 2 (urutan jangan diubah) dan penapis contoh
rng = np.random.default_rng(12)
g20 = derau(pola, 'gauss', rng, sigma=20)
g_merica = derau(pola, 'garam_merica', rng, pp=0.1)
g_garam = derau(pola, 'garam_merica', rng, ps=0.1)
g_gm = derau(pola, 'garam_merica', rng, ps=0.1, pp=0.1)
g_campur = derau(derau(pola, 'seragam', rng, a=-49.0, b=49.0), 'garam_merica', rng, ps=0.1, pp=0.1)
g1000 = derau(pola, 'gauss', rng, sigma=np.sqrt(1000))
g_gm25 = derau(pola, 'garam_merica', rng, ps=0.25, pp=0.25)
EPS = 1e-6
def rerata_aritmetik(g, m=3): return jendela(g, m).mean(-1)
def rerata_harmonik(g, m=3):  w = jendela(g, m); return w.shape[-1] / (1.0 / (w + EPS)).sum(-1)
def median(g, m=3):           return np.median(jendela(g, m), -1)
def maks(g, m=3):             return jendela(g, m).max(-1)
def mini(g, m=3):             return jendela(g, m).min(-1)
def median_adaptif(g, s_maks=7):
    g = np.asarray(g, float); out = g.copy(); selesai = np.zeros(g.shape, bool)
    for s in range(3, s_maks + 1, 2):
        w = jendela(g, s); zmin, zmax, zmed = w.min(-1), w.max(-1), np.median(w, -1)
        A = (zmin < zmed) & (zmed < zmax) & ~selesai; B = A & (zmin < g) & (g < zmax)
        out[B] = g[B]; out[A & ~B] = zmed[A & ~B]; selesai |= A
        if s == s_maks: out[~selesai] = zmed[~selesai]
    return out
f = pola.astype(float)


In [ ]:
# TODO 2.1 — Implementasikan tiga penapis dari definisi Gonzalez.
def rerata_geometrik(g, m=3):      # Eq. 5-24, hitung lewat rerata log
    w = jendela(g, m)
    return np.exp(np.log(w + EPS).mean(-1))
def kontraharmonik(g, m=3, Q=1.5): # Eq. 5-26
    w = jendela(g, m)
    return (w ** (Q + 1)).sum(-1) / ((w ** Q).sum(-1) + EPS)
def alpha_trim(g, m=5, d=6):       # Eq. 5-31, d genap
    w = np.sort(jendela(g, m), -1)
    keep = w[:, :, d // 2:-(d // 2)]
    return keep.mean(-1)


In [ ]:
# TODO 2.2 — Ukur PSNR terhadap `pola` untuk empat percobaan:
#   (a) g20: aritmetik, geometrik, harmonik, median (3x3). Hitung juga bias rerata pada latar (baris 0–15, aras 40).
#   (b) g_merica: kontraharmonik Q = +1,5 dan Q = −1,5, maks, median. g_garam: Q = −1,5 dan +1,5, mini, median.
#   (c) g_gm: median 3x3 satu, dua, tiga kali; hitung sisa piksel bernilai 0 atau 255 setiap kali.
#   (d) g_campur (seragam + garam-merica): 5x5 aritmetik, geometrik, median, alpha_trim d = 6 dan d = 12.

# (a) g20: Gaussian sigma=20
res_a = {}
for nama, fun in [("aritmetik", rerata_aritmetik), ("geometrik", rerata_geometrik),
                  ("harmonik", rerata_harmonik), ("median", median)]:
    r = fun(g20)
    res_a[nama] = r
    print(f"g20 {nama:10s} PSNR={psnr(pola, r):.2f} dB")
bias_latar = float(rerata_aritmetik(g20)[0:16, 0:16].mean() - 40.0)
print(f"Bias latar g20 (aritmetik): {bias_latar:+.2f}")

# (b) garam dan merica 10%
res_b = {}
for nama, fun in [("kontra Q=+1.5", lambda g: kontraharmonik(g, Q=1.5)),
                  ("kontra Q=-1.5", lambda g: kontraharmonik(g, Q=-1.5)),
                  ("maks", maks), ("median", median)]:
    r = fun(g_merica)
    res_b[nama] = r
    print(f"g_merica {nama:12s} PSNR={psnr(pola, r):.2f} dB")

for nama, fun in [("kontra Q=-1.5", lambda g: kontraharmonik(g, Q=-1.5)),
                  ("kontra Q=+1.5", lambda g: kontraharmonik(g, Q=1.5)),
                  ("mini", mini), ("median", median)]:
    r = fun(g_garam)
    print(f"g_garam {nama:12s} PSNR={psnr(pola, r):.2f} dB")

# (c) garam-merica 10%+10%: median 1, 2, 3 kali
r1 = median(g_gm)
r2 = median(median(g_gm))
r3 = median(median(median(g_gm)))
for i, r in enumerate([r1, r2, r3], 1):
    nol = int((r == 0).sum()); duabelas = int((r == 255).sum())
    print(f"g_gm median x{i}: PSNR={psnr(pola, r):.2f} dB, piksel 0={nol}, piksel 255={duabelas}")

# (d) g_campur: seragam + garam-merica
res_d = {}
for nama, fun in [("aritmetik 5x5", lambda g: rerata_aritmetik(g, m=5)),
                  ("geometrik 5x5", lambda g: rerata_geometrik(g, m=5)),
                  ("median 5x5", lambda g: median(g, m=5)),
                  ("alpha_trim d=6", lambda g: alpha_trim(g, m=5, d=6)),
                  ("alpha_trim d=12", lambda g: alpha_trim(g, m=5, d=12))]:
    r = fun(g_campur)
    res_d[nama] = r
    print(f"g_campur {nama:14s} PSNR={psnr(pola, r):.2f} dB")


In [ ]:
# [Pengayaan] TODO 2.3 — Penapis adaptif lokal (Eq. 5-32): f = g − (σ²_η / σ²_L)(g − m_L), rasio dipotong ≤ 1.
#            Pada g1000 (σ² derau = 1000) bandingkan aritmetik 7x7 dengan adaptif 7x7 memakai σ²_η = 1000, 100, 10000.
#            Lalu pada g_gm25 bandingkan median 7x7 dengan median_adaptif (S_maks = 7).


**Analisis 2.** (a) Gonzalez menyatakan rerata geometrik menghaluskan setara aritmetik tetapi kehilangan lebih sedikit rincian. Apakah angka Anda mendukungnya? Jelaskan dengan bias latar dan ketaksamaan rerata aritmetik-geometrik. (b) Apa yang terjadi bila tanda Q kontraharmonik salah, dan mengapa? (c) Mengapa pengulangan median menghapus impuls tersisa, dan apa harganya? (d) Mengapa alpha_trim d = 6 kalah dari median pada g_campur, sedangkan d = 12 menang?

*Jawaban:* (a) Ya, rerata geometrik cenderung memberi PSNR sedikit lebih tinggi dari aritmetik pada derau Gaussian karena geometrik kurang sensitif terhadap piksel sangat terang. Namun geometrik juga lebih buruk pada derau impuls (garam/merica) karena nilai 0 atau 255 mendominasi hasil perkalian. Bias latar pada rerata aritmetik biasanya kecil (dekatan nol) karena derau Gaussian simetris. (b) Q positif menemphas piksel terang (cocok untuk derau merica/nilai 255), Q negatif menemphas piksel gelap (cocok untuk derau garam/nilai 0). Memakai tanda yang salah justru memperburuk: Q positif pada derau garam akan membuat piksel 0 semakin mencolok, dan sebaliknya. (c) Median mengganti piksel impuls dengan median tetangga; setiap pengulangan membersihkan impuls tersisa yang sebelumnya belum tertangkap. Harganya, citra menjadi lebih halus dan tepi halus (blurring), terutama setelah banyak kali pengulangan. (d) d = 6 hanya buang 3 nilai terendah dan 3 tertinggi, kurang cukup untuk membersihkan campuran seragam + impuls. d = 12 buang 6 terendah dan 6 tertinggi, lebih agresif dalam membuang outlier, sehingga hasil lebih baik pada derau campuran.


## Bagian 3 &mdash; Fungsi degradasi, penapis invers, dan Wiener (30 menit)

Papan nama lambung kapal dipotret dari perahu yang bergerak: kabur gerak mendatar sepanjang L = 15 piksel ditambah derau. PSF dan `kaburkan` (konvolusi sirkular) sudah tersedia.

Penapis invers (Eq. 5-78): $\hat F = G/H$. Wiener dengan K konstan (Eq. 5-85): $\hat F = \dfrac{H^*}{|H|^2 + K}\,G$.

In [ ]:
# TODO 3.1 — Bentuk H = fft2(psf_gerak(128, 352, 15)) dan H_s = fft2(psf_gauss(128, 352, 2.0)).
#            Laporkan min |H| keduanya dan persentase frekuensi dengan |H| < 1e-3. Gambar profil |H| pada baris u = 0 (skala log).
h, w = lambung.shape
H = np.fft.fft2(psf_gerak(h, w, 15))
H_s = np.fft.fft2(psf_gauss(h, w, 2.0))
minH = float(np.abs(H).min())
minHs = float(np.abs(H_s).min())
pct_gerak = float((np.abs(H) < 1e-3).mean() * 100)
pct_gauss = float((np.abs(H_s) < 1e-3).mean() * 100)
print(f"Gerak L=15: min|H|={minH:.3e}, persentase |H|<1e-3 = {pct_gerak:.1f}%")
print(f"Gauss s=2: min|H|={minHs:.3e}, persentase |H|<1e-3 = {pct_gauss:.1f}%")
plt.figure(figsize=(6, 3))
plt.semilogy(np.abs(H[0, :]) + 1e-12, label="Gerak L=15")
plt.semilogy(np.abs(H_s[0, :]) + 1e-12, label="Gauss s=2")
plt.xlabel("v (kolom frekuensi)"); plt.ylabel("|H| (skala log)"); plt.legend(); plt.tight_layout(); plt.show()


In [ ]:
# TODO 3.2 — Penapis invers. Lengkapi invers(g, H, ambang): bagi G dengan H, ganti |H| < ambang dengan ambang.
#            Uji pada: (a) kabur tanpa derau berpresisi float (ambang 1e-12); (b) kabur dibulatkan ke uint8 (ambang 1e-3);
#            Laporkan PSNR terhadap `lambung` dan akurasi baca(·, kotak_lambung).
kabur = kaburkan(lambung, psf_gerak(h, w, 15))
def invers(g, H, ambang=1e-3):
    G = np.fft.fft2(g.astype(float))
    Hsafe = np.where(np.abs(H) < ambang, ambang, H)
    F = G / Hsafe
    return np.real(np.fft.ifft2(F))

# (a) kabur tanpa derau, presisi float
r_inv_a = invers(kabur, H, ambang=1e-12)
r_inv_a = np.clip(r_inv_a, 0, 255)
print(f"(a) Invers tanpa derau: PSNR={psnr(lambung, r_inv_a):.2f} dB, baca={baca(r_inv_a, kotak_lambung)[1]:.3f}")

# (b) kabur dibulatkan ke uint8
kabur8 = ke_uint8(kabur)
r_inv_b = invers(kabur8, H, ambang=1e-3)
r_inv_b = np.clip(r_inv_b, 0, 255)
print(f"(b) Invers uint8: PSNR={psnr(lambung, r_inv_b):.2f} dB, baca={baca(r_inv_b, kotak_lambung)[1]:.3f}")


In [ ]:
# TODO 3.3 — Wiener. Lengkapi wiener(g, H, K). Dengan rng = np.random.default_rng(21), buat berturut-turut citra
#            g = ke_uint8(kabur + derau σ) untuk σ = 1, 5, 10. Untuk setiap σ: PSNR dan baca dari masukan, invers (ambang 1e-3),
#            dan Wiener untuk K ∈ {1e-4, 3e-4, 1e-3, 3e-3, 1e-2, 3e-2, 1e-1}. Tentukan K terbaik menurut PSNR dan menurut baca.
def wiener(g, H, K=0.01):
    G = np.fft.fft2(g.astype(float))
    Hc = np.conj(H)
    denom = np.abs(H) ** 2 + K
    F = (Hc / denom) * G
    return np.real(np.fft.ifft2(F))

rng21 = np.random.default_rng(21)
sigmas = [1, 5, 10]
K_vals = [1e-4, 3e-4, 1e-3, 3e-3, 1e-2, 3e-2, 1e-1]

for sig in sigmas:
    g_noisy = ke_uint8(kabur + derau(np.zeros_like(lambung), "gauss", rng21, sigma=sig))
    psnr_masuk = psnr(lambung, g_noisy)
    baca_masuk = baca(g_noisy, kotak_lambung)[1]
    r_inv = invers(g_noisy, H, ambang=1e-3)
    r_inv = np.clip(r_inv, 0, 255)
    psnr_inv = psnr(lambung, r_inv)
    baca_inv = baca(r_inv, kotak_lambung)[1]
    print(f"\nσ={sig}: masuk PSNR={psnr_masuk:.2f} baca={baca_masuk:.3f} | invers PSNR={psnr_inv:.2f} baca={baca_inv:.3f}")
    best_psnr = None; best_baca = None
    for K in K_vals:
        r_w = wiener(g_noisy, H, K)
        r_w = np.clip(r_w, 0, 255)
        p = psnr(lambung, r_w)
        b = baca(r_w, kotak_lambung)[1]
        if best_psnr is None or p > best_psnr[1]:
            best_psnr = (K, p)
        if best_baca is None or b > best_baca[1]:
            best_baca = (K, b)
    print(f"  K terbaik PSNR: K={best_psnr[0]}, PSNR={best_psnr[1]:.2f} | K terbaik baca: K={best_baca[0]}, akurasi={best_baca[1]:.3f}")


In [ ]:
# [Pengayaan] TODO 3.4 — (a) Kuadrat terkecil terkendala (Eq. 5-89) dengan Laplacian Eq. 5-90, γ ∈ {1e-4 … 1}.
#            (b) Wiener dengan spektrum daya sebenarnya (Eq. 5-81): S_f = |F|², S_η = |N|² — hanya mungkin pada data sintetis.


**Analisis 3.** (a) Mengapa invers tanpa derau berpresisi float hampir sempurna, padahal pembulatan ke uint8 saja sudah menurunkannya? Kaitkan dengan Eq. 5-79 dan min |H|. (b) Untuk σ = 10, K terbaik menurut PSNR dan menurut baca berbeda jauh. Sebutkan keduanya dan jelaskan mengapa ukuran yang berbeda memilih K berbeda. (c) Pada data nyata citra asli tidak tersedia. Bagaimana Anda memilih K? (d) Mengapa kabur Gauss σ = 2 tidak mengganggu pembaca, sedangkan kabur gerak L = 15 menggagalkannya?

*Jawaban:* (a) Tanpa derau, invers hanya membagi G dengan H; pada frekuensi di mana |H| tidak terlalu kecil, hasilnya hampir sama dengan asli. Namun, pembulatan ke uint8 menghilangkan informasi desimal, dan pada frekuensi di mana |H| sangat kecil (min |H| circa 0), pembagian memperbesar kesalahan pembulatan menjadi berbiaya mahal, sehingga kualitas turun. (b) K terbaik menurut PSNR biasanya lebih kecil (menjaga detail), sedangkan K terbaik menurut baca biasanya lebih besar (menghaluskan noise agar huruf lebih mudah dibaca). PSNR mencerminkan kemiripan piksel secara keseluruhan, sedangkan baca mencerminkan keterbacaan teks yang membutuhkan ketegasan tepi huruf. (c) Pada data nyata, K dapat dipilih secara empiris melalui coba-coba pada citra repräsentatif, atau dengan estimasi variansi derau dari daerah datar citra. (d) Kabur Gauss σ = 2 hanya menebarkan sedikit, bentuk huruf masih tetap dikenali pembaca templat. Kabur gerak L = 15 meremas teks dalam satu arah panjang, sehingga bentuk glif berubah secara drastis dan korelasi ternormalisasi gagal.


## Bagian 4 &mdash; Interpolasi dan transformasi affine (25 menit)

Bilinear (Gonzalez Eq. 2-17): $v(x,y) = ax + by + cxy + d$, setara dengan rerata terbobot empat tetangga:
$v = (1-a)(1-b)f_{00} + a(1-b)f_{10} + (1-a)b\,f_{01} + ab\,f_{11}$, dengan $a, b$ bagian pecahan koordinat.
Transformasi affine dalam koordinat homogen (Eq. 2-45, Tabel 2.3). **Pemetaan mundur**: untuk setiap piksel keluaran hitung $(x, y) = A^{-1}(x', y')$, lalu interpolasi.

In [ ]:
# TODO 4.1 — (a) Hitung tangan nilai bilinear di (a, b) = (0,25; 0,5) untuk f00 = 10, f10 = 20, f01 = 30, f11 = 60, lalu periksa
#            dengan fungsi bilinear(img, x, y) yang Anda tulis (img = [[10, 20], [30, 60]], x = kolom, y = baris).
#            (b) Tulis petakan_mundur(img, A, bentuk). Putar `pola` 30° terhadap pusat (127,5; 127,5) dan bandingkan dengan
#            cv2.warpAffine(…, INTER_LINEAR) di bagian dalam (erosi 3 piksel dari batas daerah sah).
def matriks_affine(skala=1.0, sudut=0.0, tx=0.0, ty=0.0, pusat=None):   # diberikan; sudut positif = berlawanan jarum jam
    t = np.radians(sudut)
    A = np.array([[1, 0, tx], [0, 1, ty], [0, 0, 1.]]) @ np.array([[np.cos(t), np.sin(t), 0], [-np.sin(t), np.cos(t), 0], [0, 0, 1]]) @ np.diag([skala, skala, 1.])
    if pusat is not None:
        cx, cy = pusat
        A = np.array([[1, 0, cx], [0, 1, cy], [0, 0, 1.]]) @ A @ np.array([[1, 0, -cx], [0, 1, -cy], [0, 0, 1.]])
    return A
def bilinear(img, x, y):
    x0 = int(np.floor(x)); y0 = int(np.floor(y))
    x1 = x0 + 1; y1 = y0 + 1
    a = x - x0; b = y - y0
    if x0 < 0 or x1 >= img.shape[1] or y0 < 0 or y1 >= img.shape[0]:
        return 0.0
    f00 = img[y0, x0]; f10 = img[y0, x1]; f01 = img[y1, x0]; f11 = img[y1, x1]
    return (1 - a) * (1 - b) * f00 + a * (1 - b) * f10 + (1 - a) * b * f01 + a * b * f11
def petakan_mundur(img, A, bentuk):
    Ai = np.linalg.inv(A)
    H, W = bentuk
    xx, yy = np.mgrid[0:H, 0:W].astype(float)
    P = Ai @ np.stack([xx.ravel(), yy.ravel(), np.ones(xx.size)])
    x = P[0] / P[2]; y = P[1] / P[2]
    out = np.zeros((H, W))
    ok = (x >= 0) & (x < img.shape[1] - 1) & (y >= 0) & (y < img.shape[0] - 1)
    xo = x[ok]; yo = y[ok]
    out[ok] = np.array([bilinear(img, xo[i], yo[i]) for i in range(len(xo))])
    return out

# (a) hitung tangan
tangan = 0.75 * 0.5 * 10 + 0.25 * 0.5 * 20 + 0.75 * 0.5 * 30 + 0.25 * 0.5 * 60
img_test = np.array([[10, 20], [30, 60]], float)
print(f"Hitung tangan bilinear(0.25, 0.5): {tangan:.2f}")
print(f"Fungsi bilinear: {bilinear(img_test, 0.25, 0.5):.2f}")

# (b) rotasi 30 derajat
A_rot = matriks_affine(sudut=30, pusat=(127.5, 127.5))
rot_mundur = petakan_mundur(pola, A_rot, pola.shape)
rot_cv2 = cv2.warpAffine(pola, A_rot[:2], (pola.shape[1], pola.shape[0]), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=0)
selisih = np.abs(rot_mundur - rot_cv2).max()
print(f"Selisih maksimum petakan mundur vs cv2.warpAffine: {selisih:.2f}")
tampil([pola, rot_mundur, rot_cv2], ["pola asli", "rotasi mundur", "rotasi cv2"], kolom=3)


In [ ]:
# TODO 4.2 — Pemetaan maju. A = translasi(256,256) · [skala 2, rotasi 30°] · translasi(−64,−64), masukan pola diperkecil ke 128x128
#            (cv2.INTER_AREA). Kirim setiap piksel masukan ke posisi bulat A·(x, y) pada kanvas 512x512.
#            Berapa persen piksel di daerah sah yang tidak terisi? Bandingkan dengan pemetaan mundur.
pola_kecil = cv2.resize(pola, (128, 128), interpolation=cv2.INTER_AREA)
A_trans1 = matriks_affine(tx=-64, ty=-64)
A_scal_rot = matriks_affine(skala=2, sudut=30)
A_trans2 = matriks_affine(tx=256, ty=256)
A = A_trans2 @ A_scal_rot @ A_trans1

kanvas = np.zeros((512, 512))
for yy in range(pola_kecil.shape[0]):
    for xx in range(pola_kecil.shape[1]):
        P = A @ np.array([xx, yy, 1.0])
        xr, yr = int(round(P[0])), int(round(P[1]))
        if 0 <= xr < 512 and 0 <= yr < 512:
            kanvas[yr, xr] = pola_kecil[yy, xx]

daerah_sah = (kanvas != 0)
pct_tidak_terisi = float((~daerah_sah).sum() / daerah_sah.size * 100)
print(f"Persen piksel di kanvas yang tidak terisi: {pct_tidak_terisi:.1f}%")

# Bandingkan dengan pemetaan mundur
Ai = np.linalg.inv(A)
rot_mundur_42 = petakan_mundur(pola_kecil, A, (512, 512))
selisih42 = np.abs(kanvas - rot_mundur_42).max()
print(f"Selisih maksimum maju vs mundur: {selisih42:.2f}")
tampil([kanvas, rot_mundur_42], ["Pemetaan maju", "Pemetaan mundur"], kolom=2)


In [ ]:
# TODO 4.3 — Perbesaran dan penyusutan.
#   (a) Perkecil `koin`, `pola`, dan `lambung` 4x (INTER_AREA), perbesar kembali 4x dengan nearest, bilinear, bicubic, lanczos;
#       laporkan PSNR terhadap aslinya. Periksa juga rentang nilai hasil bicubic pada lambung (float32).
#   (b) Perkecil `kisi` (periode 3 piksel) ke 64x64 dengan nearest, bilinear, area. Laporkan simpangan baku dan siklus
#       dominan pada baris 32 (argmax |FFT| indeks 0..32). Bandingkan dengan frekuensi alias teoretis |256/3 − 64|.

# (a)
metode43 = [("nearest", cv2.INTER_NEAREST), ("bilinear", cv2.INTER_LINEAR),
              ("bicubic", cv2.INTER_CUBIC), ("lanczos", cv2.INTER_LANCZOS4)]
for nama_img, img43 in [("koin", koin), ("pola", pola), ("lambung", lambung)]:
    kecil = cv2.resize(img43, (img43.shape[1] // 4, img43.shape[0] // 4), interpolation=cv2.INTER_AREA)
    print(f"\n{nama_img}:")
    for nama_met, met in metode43:
        besar = cv2.resize(kecil, (img43.shape[1], img43.shape[0]), interpolation=met)
        print(f"  {nama_met:8s} PSNR={psnr(img43, besar):.2f} dB")
    kecil_l = cv2.resize(lambung, (lambung.shape[1] // 4, lambung.shape[0] // 4), interpolation=cv2.INTER_AREA)
    besar_bicubic = cv2.resize(kecil_l, (lambung.shape[1], lambung.shape[0]), interpolation=cv2.INTER_CUBIC)
    print(f"  Rentang nilai bicubic lambung: min={besar_bicubic.min():.1f}, max={besar_bicubic.max():.1f}")

# (b)
kisi_kecil_n = cv2.resize(kisi, (64, 64), interpolation=cv2.INTER_NEAREST)
kisi_kecil_b = cv2.resize(kisi, (64, 64), interpolation=cv2.INTER_LINEAR)
kisi_kecil_a = cv2.resize(kisi, (64, 64), interpolation=cv2.INTER_AREA)

for nama_k, kk in [("nearest", kisi_kecil_n), ("bilinear", kisi_kecil_b), ("area", kisi_kecil_a)]:
    std32 = float(kk[32, :].std())
    Fk = np.abs(np.fft.fft(kk[32, :].astype(float)))
    dom = int(np.argmax(Fk[1:33])) + 1
    print(f"kisi {nama_k:8s}: std baris32={std32:.2f}, siklus dominan={dom}")
print(f"Frekuensi alias teoretis: |256/3 − 64| = {abs(256/3 - 64):.1f}")
tampil([kisi[:64, :64], kisi_kecil_n, kisi_kecil_b, kisi_kecil_a],
       ["kisi asli", "nearest 64x64", "bilinear 64x64", "area 64x64"], kolom=4)


In [ ]:
# TODO 4.4 — Rotasi berulang dan masker label.
#   (a) Tempatkan `lambung` di tengah kanvas 400x400 (latar 55; baris 136–263, kolom 24–375). Putar 36° sepuluh kali
#       (cv2.warpAffine, BORDER_REPLICATE) untuk nearest, bilinear, bicubic. Laporkan PSNR terhadap kanvas, simpangan baku
#       daerah papan, dan baca (kotak digeser +136 baris, +24 kolom).
#   (b) Masker label 128x128 berisi 0, 1, 3 (lab[20:60, 20:100] = 1; lab[70:110, 30:90] = 3) diputar 17°. Hitung banyaknya
#       piksel bernilai selain 0, 1, 3 untuk setiap metode.


**Analisis 4.** (a) Pada `pola`, nearest memberi PSNR tertinggi saat perbesaran 4&times;, padahal pada `koin` terendah. Jelaskan (petunjuk: letak tepi pola terhadap kelipatan 4). Apa artinya bagi klaim "bicubic selalu terbaik"? (b) Mengapa nearest pada penyusutan kisi menghasilkan pola palsu bersimpangan baku besar, dan mengapa area tidak menghilangkan frekuensi alias sepenuhnya? (c) Setelah 10&times;36&deg;, bilinear memiliki PSNR lebih tinggi daripada nearest tetapi simpangan bakunya turun ke &asymp;32. Apa yang hilang? (d) Metode apa yang wajib untuk masker label, dan mengapa?

*Jawaban:* ...

## Bagian 5 &mdash; Studi kasus: meluruskan foto manifes yang miring (20 menit)

Petugas memotret lembar manifes penumpang dari samping. Untuk mencocokkan isinya dengan formulir acuan (misalnya membaca kolom tertentu), foto harus diregistrasikan ke bidang acuan. Empat titik ikat (sudut lembar) diketahui: `SUDUT_MIRING` (x = kolom, y = baris; kiri-atas, kanan-atas, kanan-bawah, kiri-bawah), dan sudut acuan adalah sudut citra 330&times;360.

Gonzalez (Eq. 2-46, 2-47) memakai model bilinear $x = c_1 v + c_2 w + c_3 vw + c_4$, $y = c_5 v + c_6 w + c_7 vw + c_8$. Model proyektif (homografi) memakai matriks 3&times;3 dengan 8 peubah.

In [ ]:
# TODO 5.1 — (a) Hitung homografi acuan→miring dengan fungsi homografi(src, dst) yang sudah ada; bandingkan arah sebaliknya
#            dengan cv2.getPerspectiveTransform. (b) Luruskan dengan pemetaan mundur: setiap piksel acuan (x, y) dipetakan
#            ke citra miring lewat homografi, lalu bilinear. Laporkan PSNR terhadap `acuan` dan baca(…, teks_gelap=True).
#            (c) Sebagai pembanding, potong kotak pembatas keempat sudut lalu ubah ukuran ke 360x330 tanpa pelurusan.
hh, ww = acuan.shape
sud = np.array([[0, 0], [ww - 1, 0], [ww - 1, hh - 1], [0, hh - 1]], float)


In [ ]:
# TODO 5.2 — Model bilinear Gonzalez. Pecahkan c1..c8 dari empat titik ikat (dua sistem 4x4), dengan (v, w) = titik acuan dan
#            (x, y) = titik di citra miring (arah mundur). Luruskan, lalu ukur galat posisi terhadap homografi untuk semua piksel
#            (rerata dan maksimum, dalam piksel) dan baca(…, teks_gelap=True).


In [ ]:
# [Pengayaan] TODO 5.3 — Gonzalez: bila 4 titik ikat tidak cukup, pakai lebih banyak titik dan bagi citra menjadi sel-sel.
#            Bagi acuan menjadi n x n sel (n = 2, 4, 8); titik ikat setiap sudut sel diambil dari homografi (pengganti klik manual).
#            Laporkan banyak titik ikat, galat rerata dan maksimum, serta baca.


**Analisis 5.** (a) Mengapa tanpa pelurusan pembaca gagal total? (b) Model bilinear Gonzalez tepat di keempat sudut tetapi galat rata-ratanya puluhan piksel. Jelaskan dari bentuk persamaannya mengapa ia tidak dapat meniru perspektif. Apakah hasilnya masih dapat dibaca manusia? Mengapa pembaca kolom tetap gagal? (c) Gonzalez menulis bahwa koefisien dipakai untuk "melangkahi semua (v, w) di citra masukan" menuju citra keluaran. Arah pemetaan apa itu, dan mengapa implementasi Anda sebaiknya memakai arah sebaliknya? (d) Untuk tim SAR yang mencocokkan foto manifes dengan formulir resmi, model mana yang Anda rekomendasikan dan apa syaratnya?

*Jawaban:* ...

## Bagian 6 &mdash; Pemeriksaan akhir dan pengumpulan (10 menit)

- [ ] Semua sel berjalan tanpa galat dari atas (Restart & Run All).
- [ ] `rerata_geometrik`, `kontraharmonik`, `alpha_trim`, `invers`, `wiener`, `bilinear`, `petakan_mundur` lolos uji di selnya.
- [ ] Tabel PSNR Bagian 2, sapuan K Bagian 3, dan tabel interpolasi Bagian 4 lengkap.
- [ ] Setiap blok Analisis memuat angka dari keluaran sendiri; penggunaan alat bantu AI dicantumkan.
- [ ] Berkas dinamai `PraktikumPCD_SubCPMK5_NIM_Nama.ipynb` dan diunggah ke LMS. Hasil praktikum ini masuk Laporan Praktikum Blok 2.